# 階段 6：B 組回測與三組比較

台股擇時策略研究專案第六階段。**這是本專案的核心結論。**

| 組別 | 第一層濾網 | RSI 擇時 | 進場時點 | 狀態 |
|---|---|---|---|---|
| A 買進持有 | ✗ | ✗ | 期初一次 | 階段 4 完成 |
| C 只有濾網 | ✓ | ✗ | 狀態轉多當日開盤 | 階段 4 完成 |
| **B 完整策略** | ✓ | **✓** | **RSI 訊號次日開盤** | **本階段** |

**B 與 C 的出場規則完全相同**（`regime` 轉 `FLAT` 當日開盤賣出），
唯一的差別是進場日。因此 **`B − C` 就是 RSI 這一層的淨貢獻**。

### 沿用階段 1–5 的結論（已確認，不重複檢查）

- `data/signals.csv`，6,632 筆，1999-01-05 ~ 2026-01-20
- 研究期間 25 個 `LONG_OK` 區間，**全部 25 個都產生了 B 組進場訊號**（無漏接、無作廢）
- B 組進場日 = 訊號日 + 1 個交易日，時序已驗證
- 等待交易日數：最短 3、中位數 12、平均 13.9、最長 36
- 進場價差異（B 相對 C）：中位數 +1.38%、平均 +1.62%，B 買得更便宜的比例僅 36.0%

**產出**：`data/backtest_B.csv`、`data/trades_B.csv`、`data/summary_ABC.csv`、
`data/trade_comparison_BC.csv`

---
## 研究設定與規則

研究設定與階段 4 **完全一致**（研究期間、IS/OOS 切點、曝險、報酬計算、
空手不給利息、無風險利率 0、不計交易成本、年化基準 245 交易日、不含股息）。

### B 組進出場

```
進場：signals.csv 中 b_entry_day == True 的那一天，該日開盤買進
出場：regime 由 LONG_OK 轉為 FLAT 的那一天，該日開盤賣出
```

### ★ 兩個欄位都不要再 shift

| 欄位 | 已處理的階段 | 本階段是否再 shift |
|---|---|---|
| `regime` | 階段 3（月底判斷 → 次月首日生效） | **不要** |
| `b_entry_day` | 階段 5（訊號日 → 次日成交） | **不要** |

兩者都已經是「可直接成交的那一天」，再 shift 一次會讓進出場整體延後一天。
區塊 2 會明確驗證。

### 進出場日的報酬處理（與階段 4 完全相同）

| 情境 | 公式 |
|---|---|
| 進場日 | `Close[t] / Open[t] − 1` |
| 持有中 | `Close[t] / Close[t−1] − 1` |
| 出場日 | `Open[t] / Close[t−1] − 1` |
| 空手 | `0` |

資料結束時仍持有的最後一筆標記為「持有中」，用最後一日收盤價計算。

---
## 1. 參數與載入

同時載入階段 5 的訊號檔與階段 4 的 C 組回測結果。
C 組的數字**直接沿用階段 4 的產出**，不重算 —— 但區塊 6 會重算一次做交叉驗證。

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============ 參數 ============
DATA_IN      = "data/signals.csv"
BACKTEST_C   = "data/backtest_C.csv"
TRADES_C     = "data/trades_C.csv"
DATA_OUT     = "data/backtest_B.csv"
TRADES_OUT   = "data/trades_B.csv"
SUMMARY_OUT  = "data/summary_ABC.csv"
COMPARE_OUT  = "data/trade_comparison_BC.csv"
STUDY_START  = "2000-01-01"
IS_END       = "2017-12-31"
OOS_START    = "2018-01-01"
TRADING_DAYS_PER_YEAR = 245

LONG_OK, FLAT = "LONG_OK", "FLAT"
RISK_FREE = 0.0

# ============ 前階段已確認的事實 ============
EXPECTED_ROWS   = 6632
EXPECTED_FIRST  = "1999-01-05"
EXPECTED_LAST   = "2026-01-20"
EXPECTED_BLOCKS   = 50      # 研究期間 LONG_OK 區間數（= C 組交易筆數）
EXPECTED_B_ENTRIES = 49     # B 組進場次數；區塊 34 全程無 RSI 回檔訊號，故少一筆
EXPECTED_C_LONG_DAYS = 3908

pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 400)
pd.set_option("display.max_columns", 50)
plt.rcParams["figure.figsize"] = (14, 7)

CHECKS = []

def record(name, passed, detail=""):
    verdict = "通過" if passed else "異常"
    CHECKS.append({"驗證項目": name, "結果": verdict, "說明": detail})
    print(f"===> [{verdict}] {name}" + (f"\n      {detail}" if detail else ""))

print("參數設定完成（與階段 4 一致）")
print(f"  研究期間 : {STUDY_START} ~ {EXPECTED_LAST}")
print(f"  年化基準 : {TRADING_DAYS_PER_YEAR} 交易日   無風險利率 : {RISK_FREE}")

參數設定完成（與階段 4 一致）
  研究期間 : 2000-01-01 ~ 2026-01-20
  年化基準 : 245 交易日   無風險利率 : 0.0


In [2]:
raw   = pd.read_csv(DATA_IN,    index_col="Date", parse_dates=True).sort_index()
bt_C  = pd.read_csv(BACKTEST_C, index_col="Date", parse_dates=True).sort_index()
tr_C  = pd.read_csv(TRADES_C,   index_col="#", encoding="utf-8-sig")

df = raw.loc[STUDY_START:].copy()
prev_close_first = raw["Close"].shift(1).loc[df.index[0]]

load_check = pd.DataFrame({
    "預期": [str(EXPECTED_ROWS), EXPECTED_FIRST, EXPECTED_LAST,
             str(EXPECTED_B_ENTRIES), str(EXPECTED_BLOCKS), str(EXPECTED_C_LONG_DAYS)],
    "實際": [f"{len(raw)}", f"{raw.index.min():%Y-%m-%d}", f"{raw.index.max():%Y-%m-%d}",
             f"{int(df['b_entry_day'].sum())}", f"{len(tr_C)}",
             f"{int(bt_C['position'].sum())}"],
}, index=["signals.csv 筆數", "起始日期", "結束日期",
          "b_entry_day 天數", "C 組交易筆數", "C 組 position==1 天數"])
load_check["一致"] = np.where(load_check["預期"] == load_check["實際"], "是", "★ 否")
display(load_check)

if (load_check["一致"] == "★ 否").any():
    raise RuntimeError("載入的資料與前階段不一致，請停止並人工確認。")

# 研究期間的交易日必須與階段 4 完全對齊
if not df.index.equals(bt_C.index):
    raise RuntimeError("signals.csv 與 backtest_C.csv 的交易日索引不一致。")

record("資料載入一致性", True,
       f"signals.csv {len(raw):,} 筆、backtest_C.csv {len(bt_C):,} 筆、"
       f"trades_C.csv {len(tr_C)} 筆，與階段 4、5 完全一致；"
       f"研究期間 {len(df):,} 個交易日，索引與階段 4 對齊")

,預期,實際,一致
signals.csv 筆數,6632,6632,是
起始日期,1999-01-05,1999-01-05,是
結束日期,2026-01-20,2026-01-20,是
b_entry_day 天數,49,49,是
C 組交易筆數,50,50,是
C 組 position==1 天數,3908,3908,是


===> [通過] 資料載入一致性
      signals.csv 6,632 筆、backtest_C.csv 6,391 筆、trades_C.csv 50 筆，與階段 4、5 完全一致；研究期間 6,391 個交易日，索引與階段 4 對齊


---
## 2. 建立 B 組部位序列

用明確的逐日迴圈：`b_entry_day` 當天買進，`regime` 為 `FLAT` 的那天賣出。

四項驗證：

1. B 組進場日數量 = 25，與階段 5 的 `b_entry_day` 完全一致
2. **B 組出場日與 C 組出場日完全相同** —— 這是「B 與 C 只差進場時點」的設計核心，必須嚴格驗證
3. `position_B == 1` 的天數必然**少於** C 組的 4,264 天，差額就是 RSI 造成的曝險減少
4. 目視確認 `position_B` 在 `b_entry_day` **當日**就變為 1，沒有額外 shift

In [3]:
idx     = df.index
regime  = df["regime"].to_numpy()
b_entry = df["b_entry_day"].to_numpy().astype(bool)
n       = len(df)

pos_B  = np.zeros(n, dtype=int)
in_pos = False

for t in range(n):
    if regime[t] == FLAT:      # 第一層出場：當日開盤賣出，該日部位為 0
        in_pos = False
    elif b_entry[t]:           # 第二層進場：當日開盤買進
        in_pos = True
    pos_B[t] = int(in_pos)

df["position_B"] = pos_B
df["position_C"] = bt_C["position"].to_numpy()

print(f"B 組 position==1 : {int(df['position_B'].sum()):,} 天")
print(f"C 組 position==1 : {int(df['position_C'].sum()):,} 天")
print(f"差額（RSI 造成的曝險減少）: {int(df['position_C'].sum() - df['position_B'].sum()):,} 天")

B 組 position==1 : 3,251 天
C 組 position==1 : 3,908 天
差額（RSI 造成的曝險減少）: 657 天


In [4]:
def entry_exit_days(pos):
    prev = pos.shift(1).fillna(0).astype(int)
    return pos.index[(pos == 1) & (prev == 0)], pos.index[(pos == 0) & (prev == 1)]

entB, exitB = entry_exit_days(df["position_B"])
entC, exitC = entry_exit_days(df["position_C"])

n_entry_ok  = len(entB) == EXPECTED_B_ENTRIES
entry_match = entB.equals(df.index[df["b_entry_day"]])
exit_match   = exitB.isin(exitC).all()
fewer_days   = int(df["position_B"].sum()) < int(df["position_C"].sum())

tbl = pd.DataFrame({
    "檢查": [f"B 組進場日數 = {EXPECTED_B_ENTRIES}",
             "B 組進場日 == b_entry_day 欄位",
             "★ B 組出場日 ⊆ C 組出場日",
             "B 組在場天數 < C 組在場天數"],
    "值": [f"{len(entB)}",
           f"{len(entB)} vs {int(df['b_entry_day'].sum())}",
           f"{len(exitB)} vs {len(exitC)}",
           f"{int(df['position_B'].sum()):,} vs {int(df['position_C'].sum()):,}"],
    "結果": ["通過" if n_entry_ok else "★ 異常",
             "通過" if entry_match else "★ 異常",
             "通過" if exit_match else "★ 異常",
             "通過" if fewer_days else "★ 異常"],
})
display(tbl)

pos_ok = (tbl["結果"] == "通過").all()
gap_days = int(df["position_C"].sum() - df["position_B"].sum())
record("B 組部位序列", pos_ok,
       f"進場 {len(entB)} 次（與 b_entry_day 完全一致）；"
       f"**每次出場日都與 C 組相同**（B {len(exitB)} 次 / C {len(exitC)} 次）；"
       f"在場 {int(df['position_B'].sum()):,} 天 vs C 組 {int(df['position_C'].sum()):,} 天，"
       f"少 {gap_days:,} 天（佔研究期間 {gap_days / n:.2%}），即 RSI 造成的曝險減少"
       if pos_ok else "部位序列有問題，明細見上表")

,檢查,值,結果
0,B 組進場日數 = 49,49,通過
1,B 組進場日 == b_entry_day 欄位,49 vs 49,通過
2,★ B 組出場日 ⊆ C 組出場日,48 vs 49,通過
3,B 組在場天數 < C 組在場天數,"3,251 vs 3,908",通過


===> [通過] B 組部位序列
      進場 49 次（與 b_entry_day 完全一致）；**每次出場日都與 C 組相同**（B 48 次 / C 49 次）；在場 3,251 天 vs C 組 3,908 天，少 657 天（佔研究期間 10.28%），即 RSI 造成的曝險減少


### 2a. 目視確認：`position_B` 在 `b_entry_day` 當日就變為 1

若 `position_B` 要到下一列才變成 1，代表多做了一次 shift。

In [5]:
for d in entB[:3]:
    i = df.index.get_loc(d)
    win = df.iloc[max(i - 2, 0): i + 3][
        ["Open", "Close", "regime", "b_entry_day", "position_B", "position_C"]].copy()
    win["◀"] = np.where(win.index == d, "◀ B 進場日", "")
    win.index = win.index.strftime("%Y-%m-%d")
    print(f"\n=== B 組進場 {d:%Y-%m-%d} ===")
    display(win.style.format({"Open": "{:,.2f}", "Close": "{:,.2f}"}))


=== B 組進場 2000-03-02 ===


,Open,Close,regime,b_entry_day,position_B,position_C,◀
Date,,,,,,,
2000-02-29,"9,525.65","9,435.94",LONG_OK,False,0,1,
2000-03-01,"9,572.24","9,689.10",LONG_OK,False,0,1,
2000-03-02,"9,781.27","9,543.82",LONG_OK,True,1,1,◀ B 進場日
2000-03-03,"9,557.66","9,588.03",LONG_OK,False,1,1,
2000-03-06,"9,519.80","9,367.91",LONG_OK,False,1,1,



=== B 組進場 2001-02-13 ===


,Open,Close,regime,b_entry_day,position_B,position_C,◀
Date,,,,,,,
2001-02-09,"5,782.42","5,809.84",LONG_OK,False,0,1,
2001-02-12,"5,807.61","5,847.07",LONG_OK,False,0,1,
2001-02-13,"5,922.02","6,027.49",LONG_OK,True,1,1,◀ B 進場日
2001-02-14,"6,060.89","5,887.68",LONG_OK,False,1,1,
2001-02-15,"5,963.07","6,104.24",LONG_OK,False,1,1,



=== B 組進場 2001-12-26 ===


,Open,Close,regime,b_entry_day,position_B,position_C,◀
Date,,,,,,,
2001-12-24,"5,132.61","5,164.73",LONG_OK,False,0,1,
2001-12-25,"5,198.17","5,372.81",LONG_OK,False,0,1,
2001-12-26,"5,421.73","5,392.43",LONG_OK,True,1,1,◀ B 進場日
2001-12-27,"5,464.52","5,332.98",LONG_OK,False,1,1,
2001-12-28,"5,372.85","5,398.28",LONG_OK,False,1,1,


---
## 3. 建立每日報酬序列

與階段 4 相同的四分類做法：進場／持有／出場／空手，逐類套公式。

In [6]:
prev_pos = df["position_B"].shift(1).fillna(0).astype(int)
pos      = df["position_B"]

is_entry = (pos == 1) & (prev_pos == 0)
is_hold  = (pos == 1) & (prev_pos == 1)
is_exit  = (pos == 0) & (prev_pos == 1)
is_flat  = (pos == 0) & (prev_pos == 0)

if bool((is_entry & is_exit).any()):
    raise RuntimeError("偵測到同一天既是進場日又是出場日，請停止並人工確認部位序列。")

cover = is_entry.astype(int) + is_hold.astype(int) + is_exit.astype(int) + is_flat.astype(int)
if not bool((cover == 1).all()):
    raise RuntimeError("每日分類未互斥或未涵蓋全部交易日。")

prev_close = df["Close"].shift(1)
prev_close.iloc[0] = prev_close_first

ret_B = pd.Series(0.0, index=df.index, name="ret_B")
ret_B[is_entry] = df.loc[is_entry, "Close"] / df.loc[is_entry, "Open"] - 1
ret_B[is_hold]  = df.loc[is_hold,  "Close"] / prev_close[is_hold] - 1
ret_B[is_exit]  = df.loc[is_exit,  "Open"]  / prev_close[is_exit] - 1
df["ret_B"] = ret_B

day_types = pd.DataFrame({"天數": [int(is_entry.sum()), int(is_hold.sum()),
                                   int(is_exit.sum()), int(is_flat.sum())]},
                         index=["進場日", "持有中", "出場日", "空手"])
day_types.loc["合計"] = day_types.sum()
day_types["佔比"] = (day_types["天數"] / n).map("{:.2%}".format)
display(day_types)

,天數,佔比
進場日,49,0.77%
持有中,3202,50.10%
出場日,48,0.75%
空手,3092,48.38%
合計,6391,100.00%


### 3a. 前 3 筆完整交易的逐日明細

檢查進場日用 `Close/Open − 1`、出場日用 `Open/前收 − 1`、出場後為 0。

In [7]:
exit_dates = df.index[is_exit]

for k in range(3):
    e_in = entB[k]
    later = exit_dates[exit_dates > e_in]
    e_out = later[0] if len(later) else df.index[-1]
    i0, i1 = df.index.get_loc(e_in), df.index.get_loc(e_out)
    win = df.iloc[max(i0 - 1, 0): min(i1 + 2, n)][
        ["Open", "Close", "position_B", "ret_B"]].copy()
    win["前收"] = prev_close.loc[win.index]
    win["類型"] = np.select(
        [is_entry.loc[win.index], is_hold.loc[win.index],
         is_exit.loc[win.index], is_flat.loc[win.index]],
        ["進場", "持有", "出場", "空手"], default="")
    win["ret 檢算"] = np.select(
        [win["類型"] == "進場", win["類型"] == "持有", win["類型"] == "出場"],
        [win["Close"] / win["Open"] - 1,
         win["Close"] / win["前收"] - 1,
         win["Open"] / win["前收"] - 1], default=0.0)
    win = win[["Open", "Close", "前收", "position_B", "類型", "ret_B", "ret 檢算"]]
    win.index = win.index.strftime("%Y-%m-%d")
    print(f"\n=== 第 {k + 1} 筆：{e_in:%Y-%m-%d} 進場 → {e_out:%Y-%m-%d} 出場（中間省略）===")
    fmt = {"Open": "{:,.2f}", "Close": "{:,.2f}", "前收": "{:,.2f}",
           "ret_B": "{:+.4%}", "ret 檢算": "{:+.4%}"}
    display(win.head(3).style.format(fmt))
    display(win.tail(3).style.format(fmt))


=== 第 1 筆：2000-03-02 進場 → 2000-05-02 出場（中間省略）===


,Open,Close,前收,position_B,類型,ret_B,ret 檢算
Date,,,,,,,
2000-03-01,"9,572.24","9,689.10","9,435.94",0,空手,+0.0000%,+0.0000%
2000-03-02,"9,781.27","9,543.82","9,689.10",1,進場,-2.4276%,-2.4276%
2000-03-03,"9,557.66","9,588.03","9,543.82",1,持有,+0.4632%,+0.4632%


,Open,Close,前收,position_B,類型,ret_B,ret 檢算
Date,,,,,,,
2000-04-28,"8,594.48","8,824.36","8,541.95",1,持有,+3.3062%,+3.3062%
2000-05-02,"8,836.83","8,638.75","8,824.36",0,出場,+0.1413%,+0.1413%
2000-05-03,"8,505.46","8,420.00","8,638.75",0,空手,+0.0000%,+0.0000%



=== 第 2 筆：2001-02-13 進場 → 2001-05-02 出場（中間省略）===


,Open,Close,前收,position_B,類型,ret_B,ret 檢算
Date,,,,,,,
2001-02-12,"5,807.61","5,847.07","5,809.84",0,空手,+0.0000%,+0.0000%
2001-02-13,"5,922.02","6,027.49","5,847.07",1,進場,+1.7810%,+1.7810%
2001-02-14,"6,060.89","5,887.68","6,027.49",1,持有,-2.3195%,-2.3195%


,Open,Close,前收,position_B,類型,ret_B,ret 檢算
Date,,,,,,,
2001-04-30,"5,430.59","5,381.67","5,416.67",1,持有,-0.6462%,-0.6462%
2001-05-02,"5,469.95","5,304.24","5,381.67",0,出場,+1.6404%,+1.6404%
2001-05-03,"5,287.72","5,405.54","5,304.24",0,空手,+0.0000%,+0.0000%



=== 第 3 筆：2001-12-26 進場 → 2002-05-02 出場（中間省略）===


,Open,Close,前收,position_B,類型,ret_B,ret 檢算
Date,,,,,,,
2001-12-25,"5,198.17","5,372.81","5,164.73",0,空手,+0.0000%,+0.0000%
2001-12-26,"5,421.73","5,392.43","5,372.81",1,進場,-0.5404%,-0.5404%
2001-12-27,"5,464.52","5,332.98","5,392.43",1,持有,-1.1025%,-1.1025%


,Open,Close,前收,position_B,類型,ret_B,ret 檢算
Date,,,,,,,
2002-04-30,"6,201.84","6,065.73","6,205.09",1,持有,-2.2459%,-2.2459%
2002-05-02,"6,099.27","5,867.83","6,065.73",0,出場,+0.5529%,+0.5529%
2002-05-03,"5,785.73","5,910.32","5,867.83",0,空手,+0.0000%,+0.0000%


In [8]:
flat_nonzero = int((df.loc[is_flat, "ret_B"] != 0).sum())
max_abs = df["ret_B"].abs().max()
worst_day = df["ret_B"].abs().idxmax()

ret_ok = (flat_nonzero == 0) and (max_abs <= 0.10)
record("B 組每日報酬序列", ret_ok,
       f"空手日 {int(is_flat.sum()):,} 天報酬全為 0；單日報酬最大絕對值 {max_abs:.2%}"
       f"（{worst_day:%Y-%m-%d}），未超過 ±10%"
       if ret_ok else
       f"空手日有 {flat_nonzero} 天報酬非 0，或單日報酬 {max_abs:.2%} 超過 ±10%")

===> [通過] B 組每日報酬序列
      空手日 3,092 天報酬全為 0；單日報酬最大絕對值 6.74%（2009-04-30），未超過 ±10%


---
## 4. B 組交易明細

欄位與階段 4 的 C 組相同。MAE 的定義也相同：
持有期間內收盤價相對**進場價**的最低點跌幅，取樣範圍為進場日到出場日前一天。

In [9]:
def build_trades(entry_dates, exit_dates, frame):
    """由進出場日期建出交易明細（與階段 4 相同的定義）。"""
    rows = []
    for k, e_in in enumerate(entry_dates, start=1):
        later = exit_dates[exit_dates > e_in]
        open_trade = len(later) == 0
        e_out = later[0] if not open_trade else frame.index[-1]
        i0, i1 = frame.index.get_loc(e_in), frame.index.get_loc(e_out)

        entry_px = frame.loc[e_in, "Open"]
        exit_px  = frame.loc[e_out, "Close"] if open_trade else frame.loc[e_out, "Open"]
        hold_close = frame["Close"].iloc[i0: i1 + 1] if open_trade else frame["Close"].iloc[i0: i1]

        rows.append({
            "#": k,
            "進場日": f"{e_in:%Y-%m-%d}",
            "進場價（開盤）": entry_px,
            "出場日": f"{e_out:%Y-%m-%d}" + ("（持有中）" if open_trade else ""),
            "出場價": exit_px,
            "持有交易日數": i1 - i0 + (1 if open_trade else 0),
            "報酬率": exit_px / entry_px - 1,
            "期間最大不利變動": hold_close.min() / entry_px - 1,
        })
    return pd.DataFrame(rows).set_index("#")

trades_B = build_trades(entB, exit_dates, df)

print(f"B 組共 {len(trades_B)} 筆交易")
display(trades_B.style.format({
    "進場價（開盤）": "{:,.2f}", "出場價": "{:,.2f}",
    "報酬率": "{:+.2%}", "期間最大不利變動": "{:.2%}"}))

B 組共 49 筆交易


,進場日,進場價（開盤）,出場日,出場價,持有交易日數,報酬率,期間最大不利變動
#,,,,,,,
1,2000-03-02,"9,781.27",2000-05-02,"8,836.83",40,-9.66%,-12.73%
2,2001-02-13,"5,922.02",2001-05-02,"5,469.95",53,-7.63%,-9.60%
3,2001-12-26,"5,421.73",2002-05-02,"6,099.27",80,+12.50%,-1.64%
4,2002-11-25,"4,731.31",2003-01-02,"4,460.57",27,-5.72%,-5.89%
5,2003-02-18,"4,698.10",2003-03-03,"4,483.44",8,-4.57%,-5.65%
6,2003-07-02,"5,075.21",2003-12-01,"5,768.69",106,+13.66%,0.39%
7,2004-02-09,"6,442.84",2004-04-01,"6,504.54",38,+0.96%,-4.81%
8,2004-09-22,"5,961.91",2004-11-01,"5,725.65",26,-3.96%,-5.22%
9,2005-01-12,"5,957.90",2005-04-01,"6,010.71",50,+0.89%,-3.13%


In [10]:
tr_sum = pd.DataFrame({"值": [
    f"{len(trades_B)}",
    f"{int((trades_B['報酬率'] > 0).sum())}",
    f"{(trades_B['報酬率'] > 0).mean():.1%}",
    f"{trades_B['報酬率'].mean():+.2%}",
    f"{trades_B['報酬率'].median():+.2%}",
    f"{trades_B['報酬率'].max():+.2%}",
    f"{trades_B['報酬率'].min():+.2%}",
    f"{trades_B['持有交易日數'].median():.0f}",
    f"{trades_B['期間最大不利變動'].min():.2%}",
    f"{trades_B['期間最大不利變動'].median():.2%}",
]}, index=["交易筆數", "獲利筆數", "勝率", "平均報酬", "中位數報酬", "最佳單筆",
           "最差單筆", "持有交易日數中位數", "最深單筆 MAE", "MAE 中位數"])
tr_sum.index.name = "B 組交易統計"
display(tr_sum)

record("B 組交易明細", True,
       f"{len(trades_B)} 筆交易，勝率 {(trades_B['報酬率'] > 0).mean():.1%}，"
       f"平均報酬 {trades_B['報酬率'].mean():+.2%}，"
       f"最差單筆 {trades_B['報酬率'].min():+.2%}，"
       f"最深單筆 MAE {trades_B['期間最大不利變動'].min():.2%}")

,值
B 組交易統計,
交易筆數,49
獲利筆數,21
勝率,42.9%
平均報酬,+3.17%
中位數報酬,-1.81%
最佳單筆,+56.38%
最差單筆,-11.31%
持有交易日數中位數,40
最深單筆 MAE,-14.38%


===> [通過] B 組交易明細
      49 筆交易，勝率 42.9%，平均報酬 +3.17%，最差單筆 -11.31%，最深單筆 MAE -14.38%


---
## 5. B vs C 逐筆交易對照 ★ 核心產出之一

兩組的出場日完全相同，因此 25 筆交易可以一對一並排。

- 「等待日數」= B 比 C 晚進場幾個交易日
- 「進場價差異%」= `B進場價 / C進場價 − 1`，負值代表 B 買得更便宜
- 「報酬差(B−C)」= B 的單筆報酬減 C 的單筆報酬
- 「MAE改善(B−C)」= `B的MAE − C的MAE`，**正值代表 B 的最大不利變動較小**（帳面較不痛）

MAE 本身是負值（持有期間相對進場價的最低點跌幅）。
「較不痛」等於 MAE 較接近 0、也就是**數值較大**，
所以改善量要用 `B − C` 而不是 `C − B`，正負號才會與「改善」的直覺一致。

In [11]:
tc = tr_C.copy()
tc["出場日_純"] = tc["出場日"].str.replace("（持有中）", "", regex=False)
tb = trades_B.copy()
tb["出場日_純"] = tb["出場日"].str.replace("（持有中）", "", regex=False)

# B 組可能因某個 LONG_OK 區塊內從未出現 RSI 回檔訊號而少一筆交易，
# 因此以「出場日」為鍵配對，只保留兩組都有的交易（B 的每一筆都必須對得到 C）。
if not set(tb["出場日_純"]).issubset(set(tc["出場日_純"])):
    raise RuntimeError("B 組有無法對應到 C 組出場日的交易，請停止並人工確認。")

common = [d for d in tc["出場日_純"] if d in set(tb["出場日_純"])]
tc = tc.set_index("出場日_純").loc[common]
tb = tb.set_index("出場日_純").loc[common]
n_unmatched = len(trades_B.index.union(tr_C.index)) and len(tr_C) - len(common)
if n_unmatched:
    print(f"註：C 組有 {n_unmatched} 筆交易在 B 組沒有對應（該區塊內無 RSI 進場訊號），"
          f"逐筆對照表僅含兩組都成交的 {len(common)} 筆。")

cmp = pd.DataFrame({
    "出場日（兩組相同）": tc["出場日"].to_numpy(),
    "C進場日": tc["進場日"].to_numpy(),
    "C進場價": tc["進場價（開盤）"].to_numpy(),
    "C報酬":   tc["報酬率"].to_numpy(),
    "C的MAE":  tc["期間最大不利變動"].to_numpy(),
    "B進場日": tb["進場日"].to_numpy(),
    "B進場價": tb["進場價（開盤）"].to_numpy(),
    "B報酬":   tb["報酬率"].to_numpy(),
    "B的MAE":  tb["期間最大不利變動"].to_numpy(),
}, index=pd.RangeIndex(1, len(common) + 1, name="#"))

cmp["等待日數"]      = (pd.to_datetime(tb["進場日"].to_numpy()).map(df.index.get_loc)
                        - pd.to_datetime(tc["進場日"].to_numpy()).map(df.index.get_loc))
cmp["進場價差異%"]   = cmp["B進場價"] / cmp["C進場價"] - 1
cmp["報酬差(B−C)"]   = cmp["B報酬"] - cmp["C報酬"]
# MAE 為負值（跌幅）。B 較不痛 = B 的 MAE 較接近 0 = B的MAE > C的MAE，
# 因此改善量必須是 B − C，正值才代表 B 的最大不利變動較小。
cmp["MAE改善(B−C)"]  = cmp["B的MAE"] - cmp["C的MAE"]

display(cmp.style.format({
    "C進場價": "{:,.2f}", "B進場價": "{:,.2f}",
    "C報酬": "{:+.2%}", "B報酬": "{:+.2%}",
    "C的MAE": "{:.2%}", "B的MAE": "{:.2%}",
    "進場價差異%": "{:+.2%}", "報酬差(B−C)": "{:+.2%}", "MAE改善(B−C)": "{:+.2%}"}))

註：C 組有 1 筆交易在 B 組沒有對應（該區塊內無 RSI 進場訊號），逐筆對照表僅含兩組都成交的 49 筆。


,出場日（兩組相同）,C進場日,C進場價,C報酬,C的MAE,B進場日,B進場價,B報酬,B的MAE,等待日數,進場價差異%,報酬差(B−C),MAE改善(B−C)
#,,,,,,,,,,,,,
1,2000-05-02,2000-01-04,"8,644.91",+2.22%,-1.26%,2000-03-02,"9,781.27",-9.66%,-12.73%,36,+13.14%,-11.88%,-11.47%
2,2001-05-02,2001-02-01,"5,927.25",-7.72%,-9.68%,2001-02-13,"5,922.02",-7.63%,-9.60%,8,-0.09%,+0.08%,+0.08%
3,2002-05-02,2001-12-03,"4,534.37",+34.51%,2.48%,2001-12-26,"5,421.73",+12.50%,-1.64%,17,+19.57%,-22.02%,-4.11%
4,2003-01-02,2002-11-01,"4,596.69",-2.96%,-3.14%,2002-11-25,"4,731.31",-5.72%,-5.89%,16,+2.93%,-2.76%,-2.76%
5,2003-03-03,2003-02-06,"4,975.65",-9.89%,-10.92%,2003-02-18,"4,698.10",-4.57%,-5.65%,8,-5.58%,+5.32%,+5.26%
6,2003-12-01,2003-06-02,"4,620.54",+24.85%,1.25%,2003-07-02,"5,075.21",+13.66%,0.39%,21,+9.84%,-11.18%,-0.85%
7,2004-04-01,2004-02-02,"6,379.98",+1.95%,-3.88%,2004-02-09,"6,442.84",+0.96%,-4.81%,5,+0.99%,-0.99%,-0.94%
8,2004-11-01,2004-09-01,"5,799.82",-1.28%,-2.57%,2004-09-22,"5,961.91",-3.96%,-5.22%,15,+2.79%,-2.68%,-2.65%
9,2005-04-01,2005-01-03,"6,166.39",-2.52%,-6.40%,2005-01-12,"5,957.90",+0.89%,-3.13%,7,-3.38%,+3.41%,+3.28%


In [12]:
cmp_stats = pd.DataFrame({"值": [
    f"{int((cmp['報酬差(B−C)'] > 0).sum())} / {len(cmp)}（{(cmp['報酬差(B−C)'] > 0).mean():.1%}）",
    f"{int((cmp['MAE改善(B−C)'] > 0).sum())} / {len(cmp)}（{(cmp['MAE改善(B−C)'] > 0).mean():.1%}）",
    f"{cmp['報酬差(B−C)'].median():+.2%}",
    f"{cmp['報酬差(B−C)'].mean():+.2%}",
    f"{cmp['報酬差(B−C)'].max():+.2%}（#{int(cmp['報酬差(B−C)'].idxmax())}）",
    f"{cmp['報酬差(B−C)'].min():+.2%}（#{int(cmp['報酬差(B−C)'].idxmin())}）",
    f"{cmp['MAE改善(B−C)'].median():+.2%}",
    f"{cmp['MAE改善(B−C)'].mean():+.2%}",
]}, index=["B 報酬優於 C 的筆數", "B 的 MAE 優於 C 的筆數",
           "報酬差 中位數", "報酬差 平均", "報酬差 最好", "報酬差 最差",
           "MAE 改善 中位數", "MAE 改善 平均"])
cmp_stats.index.name = "B vs C 逐筆統計"
display(cmp_stats)

record("B vs C 逐筆對照", True,
       f"25 筆一對一對照（出場日完全相同）：B 報酬優於 C 者 "
       f"{int((cmp['報酬差(B−C)'] > 0).sum())} 筆（{(cmp['報酬差(B−C)'] > 0).mean():.1%}），"
       f"報酬差中位數 {cmp['報酬差(B−C)'].median():+.2%}、平均 {cmp['報酬差(B−C)'].mean():+.2%}；"
       f"B 的 MAE 優於 C 者 {int((cmp['MAE改善(B−C)'] > 0).sum())} 筆，"
       f"MAE 改善中位數 {cmp['MAE改善(B−C)'].median():+.2%}")

,值
B vs C 逐筆統計,
B 報酬優於 C 的筆數,18 / 49（36.7%）
B 的 MAE 優於 C 的筆數,22 / 49（44.9%）
報酬差 中位數,-1.34%
報酬差 平均,-1.92%
報酬差 最好,+5.83%（#42）
報酬差 最差,-22.02%（#3）
MAE 改善 中位數,-0.07%
MAE 改善 平均,-0.57%


===> [通過] B vs C 逐筆對照
      25 筆一對一對照（出場日完全相同）：B 報酬優於 C 者 18 筆（36.7%），報酬差中位數 -1.34%、平均 -1.92%；B 的 MAE 優於 C 者 22 筆，MAE 改善中位數 -0.07%


---
## 6. 權益曲線與績效指標

`perf_stats()` 與階段 4 **完全相同**，直接複製過來未做任何修改 ——
三組指標的計算方式必須一致，否則比較無效。

接著重算 A、C 兩組並與階段 4 的產出比對：
不只比對彙總數字，而是**逐日比對整條權益曲線與回撤序列**，
差異必須小於 1e-10。這能抓出任何函式或資料的意外變動。

In [13]:
# ===== 以下三個函式與階段 4 完全相同，未做任何修改 =====
def drawdown(equity):
    """回傳回撤序列（相對歷史高點）。"""
    return equity / equity.cummax() - 1

def dd_detail(equity):
    """最大回撤的幅度、起始（前高）日、谷底日、恢復日。"""
    dd = drawdown(equity)
    trough = dd.idxmin()
    mdd = dd.loc[trough]
    peak = equity.loc[:trough].idxmax()
    after = equity.loc[trough:]
    rec = after[after >= equity.loc[peak]]
    recovered = rec.index[0] if len(rec) else None
    return mdd, peak, trough, recovered

def perf_stats(returns, label, position=None, trade_rets=None, trade_equity=None):
    """由日報酬序列計算績效指標。無風險利率 = 0。"""
    r = returns.dropna()
    eq = (1 + r).cumprod()
    n = len(r)

    total  = eq.iloc[-1] - 1
    cagr   = eq.iloc[-1] ** (TRADING_DAYS_PER_YEAR / n) - 1
    vol    = r.std(ddof=1) * np.sqrt(TRADING_DAYS_PER_YEAR)
    sharpe = (cagr - RISK_FREE) / vol if vol > 0 else np.nan

    mdd, peak, trough, recovered = dd_detail(eq)
    calmar = cagr / abs(mdd) if mdd < 0 else np.nan

    mdd_trade = np.nan
    if trade_equity is not None and len(trade_equity) > 1:
        mdd_trade = drawdown(trade_equity).min()

    out = {
        "總報酬率": total,
        "年化報酬率": cagr,
        "年化波動度": vol,
        "Sharpe": sharpe,
        "MDD（每日）": mdd,
        "MDD（交易層級）": mdd_trade,
        "MDD 起始日": f"{peak:%Y-%m-%d}",
        "MDD 谷底日": f"{trough:%Y-%m-%d}",
        "MDD 恢復日": f"{recovered:%Y-%m-%d}" if recovered is not None else "尚未恢復",
        "Calmar": calmar,
        "在場時間比例": (position == 1).mean() if position is not None else 1.0,
        "交易次數": len(trade_rets) if trade_rets is not None else 1,
        "勝率": (np.asarray(trade_rets) > 0).mean() if trade_rets is not None and len(trade_rets) else np.nan,
        "交易日數": n,
    }
    return pd.Series(out, name=label)

def trade_equity_curve(trade_returns):
    """由每筆交易報酬構成的權益序列，起點 1.0。"""
    return pd.Series(np.concatenate([[1.0], np.cumprod(1 + np.asarray(trade_returns))]))

print("perf_stats / drawdown / dd_detail / trade_equity_curve 已定義（與階段 4 相同）")

perf_stats / drawdown / dd_detail / trade_equity_curve 已定義（與階段 4 相同）


In [14]:
ret_A = bt_C["ret_A"]
ret_C = bt_C["ret"]

eq_A = (1 + ret_A).cumprod().rename("equity_A")
eq_C = (1 + ret_C).cumprod().rename("equity_C")
eq_B = (1 + df["ret_B"]).cumprod().rename("equity_B")

dd_A, dd_C, dd_B = drawdown(eq_A), drawdown(eq_C), drawdown(eq_B)

# 與階段 4 存檔的序列逐日比對
diffs = {
    "equity_A": np.abs(eq_A - bt_C["equity_A"]).max(),
    "equity_C": np.abs(eq_C - bt_C["equity_C"]).max(),
    "drawdown_A": np.abs(dd_A - bt_C["drawdown_A"]).max(),
    "drawdown_C": np.abs(dd_C - bt_C["drawdown_C"]).max(),
}
rep = pd.DataFrame({"與階段 4 的最大逐日差異": pd.Series(diffs)})
rep["結果"] = np.where(rep["與階段 4 的最大逐日差異"] < 1e-10, "通過", "★ 異常")
display(rep.style.format({"與階段 4 的最大逐日差異": "{:.3e}"}))

repro_ok = bool((rep["結果"] == "通過").all())
if not repro_ok:
    raise RuntimeError("重算的 A／C 組與階段 4 不一致，請停止並回報。")

record("A／C 組重現階段 4 結果", repro_ok,
       f"逐日比對 equity_A、equity_C、drawdown_A、drawdown_C 四條序列"
       f"（各 {len(df):,} 天），最大差異 {max(diffs.values()):.2e}，小於 1e-10")

,與階段 4 的最大逐日差異,結果
equity_A,5.187e-13,通過
equity_C,5.986e-13,通過
drawdown_A,9.265e-14,通過
drawdown_C,2.708e-14,通過


===> [通過] A／C 組重現階段 4 結果
      逐日比對 equity_A、equity_C、drawdown_A、drawdown_C 四條序列（各 6,391 天），最大差異 5.99e-13，小於 1e-10


---
## 7. 三組績效比較表 ★ 本專案核心產出

A、C、B 三組 × 全期間／IS／OOS。

**IS 與 OOS 的權益曲線各自從 1.0 重新起算**，
因此兩段的總報酬率不能相加，也不能與全期間直接比較 ——
每一段都是獨立的「假設從這一天開始投入 1 元」的結果。

In [15]:
WINDOWS = [("全期間", STUDY_START, f"{df.index[-1]:%Y-%m-%d}"),
           ("IS",     STUDY_START, IS_END),
           ("OOS",    OOS_START,   f"{df.index[-1]:%Y-%m-%d}")]

def trades_in(tbl, a, b):
    d = pd.to_datetime(tbl["進場日"])
    return tbl[(d >= pd.Timestamp(a)) & (d <= pd.Timestamp(b))]

blocks = []
for wname, a, b in WINDOWS:
    tC = trades_in(tr_C, a, b)
    tB = trades_in(trades_B, a, b)

    sA = perf_stats(ret_A.loc[a:b], "A 買進持有")
    sC = perf_stats(ret_C.loc[a:b], "C 只有濾網",
                    position=df["position_C"].loc[a:b],
                    trade_rets=tC["報酬率"].to_numpy(),
                    trade_equity=trade_equity_curve(tC["報酬率"].to_numpy()))
    sB = perf_stats(df["ret_B"].loc[a:b], "B 完整策略",
                    position=df["position_B"].loc[a:b],
                    trade_rets=tB["報酬率"].to_numpy(),
                    trade_equity=trade_equity_curve(tB["報酬率"].to_numpy()))
    blk = pd.concat([sA, sC, sB], axis=1)
    blk.columns = pd.MultiIndex.from_product([[wname], blk.columns])
    blocks.append(blk)

perf = pd.concat(blocks, axis=1)

PCT_ROWS = ["總報酬率", "年化報酬率", "年化波動度", "MDD（每日）",
            "MDD（交易層級）", "在場時間比例", "勝率"]
NUM_ROWS = ["Sharpe", "Calmar"]

def fmt_perf(v, row):
    if pd.isna(v):
        return "—"
    if row in PCT_ROWS:
        return f"{v:.2%}"
    if row in NUM_ROWS:
        return f"{v:.2f}"
    if row in ("交易次數", "交易日數"):
        return f"{int(v):,}"
    return v

perf_disp = perf.astype(object)
for r in perf_disp.index:
    perf_disp.loc[r] = [fmt_perf(v, r) for v in perf.loc[r]]
display(perf_disp)

全期間                                  IS                                 OOS                        
               A 買進持有      C 只有濾網      B 完整策略      A 買進持有      C 只有濾網      B 完整策略      A 買進持有      C 只有濾網      B 完整策略
總報酬率          267.38%     622.53%     214.14%      23.11%     226.32%      73.75%     198.42%     121.42%      80.80%
年化報酬率           5.11%       7.88%       4.49%       1.16%       6.75%       3.10%      14.67%      10.46%       7.70%
年化波動度          20.52%      13.40%      12.18%      21.50%      13.69%      12.27%      18.11%      12.72%      11.96%
Sharpe           0.25        0.59        0.37        0.05        0.49        0.25        0.81        0.82        0.64
MDD（每日）       -66.22%     -30.38%     -34.24%     -66.22%     -25.73%     -34.24%     -31.63%     -30.38%     -23.71%
MDD（交易層級）           —     -22.21%     -27.92%           —     -16.92%     -27.92%           —     -22.21%     -17.03%
MDD 起始日    2000-02-17  2021-07-15  2009-10-20  2000-02-17  2010-01-15  2009-10-20  2022-01-04  2021-07-15  2021-07-15
MDD 谷底日    2001-10-03  2022-12-29  2012-07-26  2001-10-03  2012-07-26  2012-07-26  2022-10-25  2022-12-29  2022-12-29
MDD 恢復日    2017-06-05  2024-06-14  2021-02-17  2017-06-05  2017-03-16        尚未恢復  2024-02-15  2024-06-14  2024-05-16
Calmar           0.08        0.26        0.13        0.02        0.26        0.09        0.46        0.34        0.32
在場時間比例        100.00%      61.15%      50.87%     100.00%      59.79%      50.25%     100.00%      64.23%      52.27%
交易次數                1          50          49           1          32          32           1          18          17
勝率                  —      52.00%      42.86%           —      46.88%      43.75%           —      61.11%      41.18%
交易日數            6,391       6,391       6,391       4,434       4,434       4,434       1,957       1,957       1,957

In [16]:
record("三組績效比較表", True,
       "；".join(
           f"{w} 年化報酬 A {perf[(w, 'A 買進持有')]['年化報酬率']:.2%} / "
           f"C {perf[(w, 'C 只有濾網')]['年化報酬率']:.2%} / "
           f"B {perf[(w, 'B 完整策略')]['年化報酬率']:.2%}"
           for w in ["全期間", "IS", "OOS"]))

===> [通過] 三組績效比較表
      全期間 年化報酬 A 5.11% / C 7.88% / B 4.49%；IS 年化報酬 A 1.16% / C 6.75% / B 3.10%；OOS 年化報酬 A 14.67% / C 10.46% / B 7.70%


---
## 8. 貢獻分解 ★ 核心產出之二

把兩層的貢獻拆開：

- **濾網貢獻 = C − A** —— 加上第一層 MA50 月度濾網帶來的變化
- **RSI 貢獻 = B − C** —— 在濾網之上再加第二層 RSI 擇時帶來的變化

因為 B 與 C 的出場規則完全相同、只差進場日，所以 `B − C` 是乾淨的 RSI 效果。

**這張表只呈現數字，不下判斷、不評價。**

In [17]:
ROWS = ["年化報酬率", "Sharpe", "MDD（每日）", "Calmar", "年化波動度", "在場時間比例"]

def contrib_table(w):
    A = perf[(w, "A 買進持有")]; C = perf[(w, "C 只有濾網")]; B = perf[(w, "B 完整策略")]
    t = pd.DataFrame({
        "A": [A[r] for r in ROWS],
        "C": [C[r] for r in ROWS],
        "B": [B[r] for r in ROWS],
        "濾網貢獻 (C−A)": [C[r] - A[r] for r in ROWS],
        "RSI貢獻 (B−C)":  [B[r] - C[r] for r in ROWS],
    }, index=ROWS)
    t.index.name = f"{w}"
    return t

def fmt_contrib(t):
    out = t.astype(object)
    for r in t.index:
        if r in ("Sharpe", "Calmar"):
            out.loc[r] = [f"{v:+.2f}" if c.endswith("）") else f"{v:.2f}"
                          for c, v in zip(t.columns, t.loc[r])]
        else:
            out.loc[r] = [f"{v:+.2%}" if c.endswith("）") else f"{v:.2%}"
                          for c, v in zip(t.columns, t.loc[r])]
    return out

contribs = {}
for w in ["全期間", "IS", "OOS"]:
    t = contrib_table(w)
    contribs[w] = t
    print(f"\n=== {w} ===")
    display(fmt_contrib(t))


=== 全期間 ===


,A,C,B,濾網貢獻 (C−A),RSI貢獻 (B−C)
全期間,,,,,
年化報酬率,5.11%,7.88%,4.49%,2.76%,-3.39%
Sharpe,0.25,0.59,0.37,0.34,-0.22
MDD（每日）,-66.22%,-30.38%,-34.24%,35.84%,-3.86%
Calmar,0.08,0.26,0.13,0.18,-0.13
年化波動度,20.52%,13.40%,12.18%,-7.12%,-1.22%
在場時間比例,100.00%,61.15%,50.87%,-38.85%,-10.28%



=== IS ===


,A,C,B,濾網貢獻 (C−A),RSI貢獻 (B−C)
IS,,,,,
年化報酬率,1.16%,6.75%,3.10%,5.60%,-3.65%
Sharpe,0.05,0.49,0.25,0.44,-0.24
MDD（每日）,-66.22%,-25.73%,-34.24%,40.49%,-8.51%
Calmar,0.02,0.26,0.09,0.25,-0.17
年化波動度,21.50%,13.69%,12.27%,-7.81%,-1.42%
在場時間比例,100.00%,59.79%,50.25%,-40.21%,-9.54%



=== OOS ===


,A,C,B,濾網貢獻 (C−A),RSI貢獻 (B−C)
OOS,,,,,
年化報酬率,14.67%,10.46%,7.70%,-4.21%,-2.77%
Sharpe,0.81,0.82,0.64,0.01,-0.18
MDD（每日）,-31.63%,-30.38%,-23.71%,1.25%,6.68%
Calmar,0.46,0.34,0.32,-0.12,-0.02
年化波動度,18.11%,12.72%,11.96%,-5.39%,-0.75%
在場時間比例,100.00%,64.23%,52.27%,-35.77%,-11.96%


In [18]:
record("貢獻分解", True,
       "；".join(
           f"{w}：濾網貢獻（年化）{contribs[w].loc['年化報酬率', '濾網貢獻 (C−A)']:+.2%}、"
           f"RSI 貢獻（年化）{contribs[w].loc['年化報酬率', 'RSI貢獻 (B−C)']:+.2%}"
           for w in ["全期間", "IS", "OOS"]))

===> [通過] 貢獻分解
      全期間：濾網貢獻（年化）+2.76%、RSI 貢獻（年化）-3.39%；IS：濾網貢獻（年化）+5.60%、RSI 貢獻（年化）-3.65%；OOS：濾網貢獻（年化）-4.21%、RSI 貢獻（年化）-2.77%


---
## 9. RSI 效果拆解 ★ 核心產出之三

RSI 延後進場同時造成兩種**方向相反**的效果，這一格把它們分開量化：

- **(a) 錯過的行情（成本）** —— C 進場日到 B 進場日之間的指數變動。
  正值代表 B 錯過了漲幅（成本），負值代表 B 躲過了跌幅（收益）。
- **(b) 避開的帳面回撤（收益）** —— `B的MAE − C的MAE`，
  即 B 因為晚進場而少承受的帳面痛苦。MAE 為負值，正值代表 B 較不痛。
- **(c) 淨效果** —— 把 25 筆交易依這兩個維度分成四個象限。

In [19]:
# (a) 錯過的行情
miss = cmp["進場價差異%"]
a_tbl = pd.DataFrame({"值": [
    f"{int((miss > 0).sum())} / {len(miss)}（{(miss > 0).mean():.1%}）",
    f"{int((miss < 0).sum())} / {len(miss)}（{(miss < 0).mean():.1%}）",
    f"{miss.median():+.2%}", f"{miss.mean():+.2%}",
    f"{miss.max():+.2%}（#{int(miss.idxmax())}）",
    f"{miss.min():+.2%}（#{int(miss.idxmin())}）",
    f"{miss.sum():+.2%}",
]}, index=["錯過漲幅的筆數（差異 > 0）", "躲過跌幅的筆數（差異 < 0）",
           "中位數", "平均", "最大（錯過最多）", "最小（躲過最多）", "單純加總"])
a_tbl.index.name = "(a) 等待期間的指數變動"
display(a_tbl)

# (b) 避開的帳面回撤
imp = cmp["MAE改善(B−C)"]
b_tbl = pd.DataFrame({"值": [
    f"{int((imp > 0).sum())} / {len(imp)}（{(imp > 0).mean():.1%}）",
    f"{imp.median():+.2%}", f"{imp.mean():+.2%}",
    f"{imp.max():+.2%}（#{int(imp.idxmax())}）",
    f"{imp.min():+.2%}（#{int(imp.idxmin())}）",
    f"{cmp['C的MAE'].mean():.2%} → {cmp['B的MAE'].mean():.2%}",
]}, index=["MAE 改善的筆數（> 0）", "中位數", "平均",
           "最大改善", "最大惡化", "平均 MAE：C → B"])
b_tbl.index.name = "(b) 避開的帳面回撤"
display(b_tbl)

,值
(a) 等待期間的指數變動,
錯過漲幅的筆數（差異 > 0）,31 / 49（63.3%）
躲過跌幅的筆數（差異 < 0）,18 / 49（36.7%）
中位數,+1.38%
平均,+1.81%
最大（錯過最多）,+19.57%（#3）
最小（躲過最多）,-6.17%（#42）
單純加總,+88.79%


,值
(b) 避開的帳面回撤,
MAE 改善的筆數（> 0）,22 / 49（44.9%）
中位數,-0.07%
平均,-0.57%
最大改善,+5.83%（#42）
最大惡化,-11.47%（#1）
平均 MAE：C → B,-3.65% → -4.22%


In [20]:
# (c) 四象限
q = pd.Series(index=cmp.index, dtype=object)
q[(miss > 0) & (imp > 0)]  = "錯過漲幅，但避開回撤"
q[(miss > 0) & (imp <= 0)] = "錯過漲幅，且未避開回撤（純損失）"
q[(miss <= 0) & (imp > 0)] = "躲過跌幅，且避開回撤（純收益）"
q[(miss <= 0) & (imp <= 0)] = "躲過跌幅，但 MAE 反而更差"
cmp["象限"] = q

quad = q.value_counts().to_frame("筆數")
quad["比例"] = (quad["筆數"] / len(cmp)).map("{:.1%}".format)
quad["平均報酬差(B−C)"] = cmp.groupby("象限")["報酬差(B−C)"].mean().map("{:+.2%}".format)
quad.index.name = "象限"
display(quad)

record("RSI 效果拆解", True,
       f"(a) 等待期間指數變動：錯過漲幅 {int((miss > 0).sum())} 筆、"
       f"躲過跌幅 {int((miss < 0).sum())} 筆，中位數 {miss.median():+.2%}；"
       f"(b) MAE 改善 {int((imp > 0).sum())} 筆，平均 {imp.mean():+.2%}"
       f"（平均 MAE {cmp['C的MAE'].mean():.2%} → {cmp['B的MAE'].mean():.2%}）；"
       f"(c) 純損失 {int((q == '錯過漲幅，且未避開回撤（純損失）').sum())} 筆、"
       f"純收益 {int((q == '躲過跌幅，且避開回撤（純收益）').sum())} 筆")

,筆數,比例,平均報酬差(B−C)
象限,,,
錯過漲幅，且未避開回撤（純損失）,27,55.1%,-4.31%
躲過跌幅，且避開回撤（純收益）,18,36.7%,+1.81%
錯過漲幅，但避開回撤,4,8.2%,-2.61%


===> [通過] RSI 效果拆解
      (a) 等待期間指數變動：錯過漲幅 31 筆、躲過跌幅 18 筆，中位數 +1.38%；(b) MAE 改善 22 筆，平均 -0.57%（平均 MAE -3.65% → -4.22%）；(c) 純損失 27 筆、純收益 18 筆


---
## 10. 逐年報酬三組對照

標記每年報酬最高的組別。2026 年只有 13 個交易日，屬部分年度。

In [21]:
def yearly(r):
    return r.groupby(r.index.year).apply(lambda s: (1 + s).prod() - 1)

yr = pd.DataFrame({"A 買進持有": yearly(ret_A),
                   "C 只有濾網": yearly(ret_C),
                   "B 完整策略": yearly(df["ret_B"])})
yr["最佳組別"] = yr[["A 買進持有", "C 只有濾網", "B 完整策略"]].idxmax(axis=1)
yr["B − C"] = yr["B 完整策略"] - yr["C 只有濾網"]
yr["B 在場比例"] = df["position_B"].groupby(df.index.year).mean()
yr["交易日數"] = df["ret_B"].groupby(df.index.year).size()
yr["備註"] = np.where(yr["交易日數"] < 200, "部分年度", "")
yr.index.name = "年份"

display(yr.style.format({"A 買進持有": "{:+.2%}", "C 只有濾網": "{:+.2%}",
                         "B 完整策略": "{:+.2%}", "B − C": "{:+.2%}",
                         "B 在場比例": "{:.0%}"}))

,A 買進持有,C 只有濾網,B 完整策略,最佳組別,B − C,B 在場比例,交易日數,備註
年份,,,,,,,,
2000,-45.12%,+2.22%,-9.66%,C 只有濾網,-11.88%,16%,245,
2001,+17.02%,+12.98%,-5.43%,A 買進持有,-18.41%,23%,245,
2002,-19.79%,+6.42%,+3.40%,C 只有濾網,-3.03%,42%,248,
2003,+32.30%,+12.70%,+8.67%,A 買進持有,-4.03%,46%,249,
2004,+4.23%,+0.65%,-3.04%,A 買進持有,-3.69%,26%,250,
2005,+6.66%,+4.37%,+0.34%,A 買進持有,-4.03%,41%,247,
2006,+19.48%,+23.85%,+21.06%,C 只有濾網,-2.79%,69%,247,
2007,+8.72%,+4.37%,+2.83%,A 買進持有,-1.54%,79%,243,
2008,-46.03%,+5.17%,+5.97%,B 完整策略,+0.80%,20%,249,


In [22]:
full = yr[yr["備註"] == ""]
down = full[full["A 買進持有"] < 0]
up   = full[full["A 買進持有"] >= 0]

def grp(g, label):
    return pd.Series({
        "年數": len(g),
        "A 平均": g["A 買進持有"].mean(),
        "C 平均": g["C 只有濾網"].mean(),
        "B 平均": g["B 完整策略"].mean(),
        "B − C 平均": g["B − C"].mean(),
        "B 最佳的年數": int((g["最佳組別"] == "B 完整策略").sum()),
    }, name=label)

grp_tbl = pd.concat([grp(down, "A 下跌年"), grp(up, "A 上漲年"),
                     grp(full, "全部完整年度")], axis=1)
display(grp_tbl.style.format({
    "年數": "{:.0f}", "A 平均": "{:+.2%}", "C 平均": "{:+.2%}",
    "B 平均": "{:+.2%}", "B − C 平均": "{:+.2%}", "B 最佳的年數": "{:.0f}"}))

best = yr["最佳組別"].value_counts()
record("逐年報酬三組對照", True,
       f"完整年度 {len(full)} 年：A 下跌的 {len(down)} 年三組平均 "
       f"A {down['A 買進持有'].mean():+.2%} / C {down['C 只有濾網'].mean():+.2%} / "
       f"B {down['B 完整策略'].mean():+.2%}；"
       f"A 上漲的 {len(up)} 年 A {up['A 買進持有'].mean():+.2%} / "
       f"C {up['C 只有濾網'].mean():+.2%} / B {up['B 完整策略'].mean():+.2%}；"
       f"各組奪冠年數 " + "、".join(f"{k} {v}" for k, v in best.items()))

,A 下跌年,A 上漲年,全部完整年度
年數,7.000000,19.000000,26.000000
A 平均,-0.247914,0.201024,0.080156
C 平均,-0.054666,0.138561,0.086538
B 平均,-0.066345,0.093598,0.050537
B − C 平均,-0.011679,-0.044962,-0.036002
B 最佳的年數,3.000000,0.000000,3.000000


===> [通過] 逐年報酬三組對照
      完整年度 26 年：A 下跌的 7 年三組平均 A -24.79% / C -5.47% / B -6.63%；A 上漲的 19 年 A +20.10% / C +13.86% / B +9.36%；各組奪冠年數 A 買進持有 19、C 只有濾網 5、B 完整策略 3


---
## 11. 視覺化

### 圖 1｜三組權益曲線（對數座標）

灰色背景為 `regime == FLAT` 區間，虛線為 IS/OOS 分界。

In [23]:
def shade_flat(ax, frame):
    s = (frame["regime"] == LONG_OK).astype(int)
    gid = (s != s.shift(1)).cumsum()
    for _, seg in s.groupby(gid):
        if seg.iloc[0] == 0:
            i = frame.index.get_loc(seg.index[-1])
            end = frame.index[min(i + 1, len(frame) - 1)]
            ax.axvspan(seg.index[0], end, color="grey", alpha=0.20, lw=0)

fig, ax = plt.subplots(figsize=(15, 7))
ax.plot(eq_A.index, eq_A, lw=1.0, color="#8a8a8a", label="A: Buy & Hold")
ax.plot(eq_C.index, eq_C, lw=1.3, color="#3b6ea5", label="C: Filter only")
ax.plot(eq_B.index, eq_B, lw=1.3, color="#b3261e", label="B: Filter + RSI")
shade_flat(ax, df)
ax.axvline(pd.Timestamp(OOS_START), color="black", ls="--", lw=1.2)
ax.text(pd.Timestamp(OOS_START), eq_A.max(), "  IS | OOS", va="top", ha="left", fontsize=10)
ax.set_yscale("log")
ax.set_title("Equity curves (start = 1.0, log scale) — shaded = regime FLAT")
ax.set_xlabel("Date"); ax.set_ylabel("Equity (log)")
ax.legend(loc="upper left"); ax.grid(alpha=0.3, which="both")
plt.tight_layout(); plt.show()

C:\Users\king5\AppData\Local\Temp\ipykernel_5136\3843308979.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


### 圖 2｜三組回撤曲線（每日權益）

In [24]:
fig, ax = plt.subplots(figsize=(15, 6))
ax.fill_between(dd_A.index, dd_A * 100, 0, color="#8a8a8a", alpha=0.50, label="A: Buy & Hold")
ax.fill_between(dd_C.index, dd_C * 100, 0, color="#3b6ea5", alpha=0.50, label="C: Filter only")
ax.fill_between(dd_B.index, dd_B * 100, 0, color="#b3261e", alpha=0.45, label="B: Filter + RSI")
ax.axvline(pd.Timestamp(OOS_START), color="black", ls="--", lw=1.2)
ax.set_title("Drawdown from daily equity curve")
ax.set_xlabel("Date"); ax.set_ylabel("Drawdown (%)")
ax.legend(loc="lower left"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

C:\Users\king5\AppData\Local\Temp\ipykernel_5136\2163434147.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


### 圖 3｜B − C 累積差異

`equity_B / equity_C` 的比值序列。高於 1 代表 RSI 這一層累積至今是加分，
低於 1 代表是扣分。用來看拖累（或貢獻）是持續累積還是集中在某幾段。

In [25]:
ratio = (eq_B / eq_C).rename("B/C")

fig, ax = plt.subplots(figsize=(15, 6))
ax.plot(ratio.index, ratio, lw=1.2, color="#6a4c93")
ax.axhline(1.0, color="black", ls="--", lw=1.0)
ax.fill_between(ratio.index, ratio, 1.0, where=(ratio >= 1),
                color="#2e7d32", alpha=0.35, label="B ahead of C")
ax.fill_between(ratio.index, ratio, 1.0, where=(ratio < 1),
                color="#b3261e", alpha=0.35, label="B behind C")
ax.axvline(pd.Timestamp(OOS_START), color="black", ls="--", lw=1.2)
ax.set_title("Cumulative effect of the RSI layer: equity_B / equity_C")
ax.set_xlabel("Date"); ax.set_ylabel("Ratio")
ax.legend(loc="best"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f"期末比值 {ratio.iloc[-1]:.4f}"
      f"（B 相對 C 累積 {ratio.iloc[-1] - 1:+.2%}）；"
      f"期間最高 {ratio.max():.4f}（{ratio.idxmax():%Y-%m-%d}）、"
      f"最低 {ratio.min():.4f}（{ratio.idxmin():%Y-%m-%d}）")

期末比值 0.4348（B 相對 C 累積 -56.52%）；期間最高 0.9873（2000-01-04）、最低 0.4331（2021-12-09）


C:\Users\king5\AppData\Local\Temp\ipykernel_5136\3070785995.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


### 圖 4｜逐筆交易報酬對照

In [26]:
x = np.arange(len(cmp))
w = 0.4
fig, ax = plt.subplots(figsize=(15, 6))
ax.bar(x - w / 2, cmp["C報酬"] * 100, w, color="#3b6ea5", label="C: Filter only")
ax.bar(x + w / 2, cmp["B報酬"] * 100, w, color="#b3261e", label="B: Filter + RSI")
ax.axhline(0, color="black", lw=0.8)
ax.set_xticks(x)
ax.set_xticklabels([f"#{i}\n{d[:7]}" for i, d in zip(cmp.index, cmp["C進場日"])],
                   fontsize=7, rotation=45)
ax.set_title("Per-trade return: B vs C (same exit dates, different entries)")
ax.set_xlabel("Trade"); ax.set_ylabel("Return (%)")
ax.legend(); ax.grid(alpha=0.3, axis="y")
plt.tight_layout(); plt.show()

C:\Users\king5\AppData\Local\Temp\ipykernel_5136\1327898603.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


### 圖 5｜RSI 效果散布圖

橫軸 = 進場價差異%（右邊代表 B 錯過漲幅），
縱軸 = MAE 改善%（上方代表 B 較不痛）。四個象限用不同顏色。

In [27]:
COLORS = {
    "錯過漲幅，但避開回撤":            "#e08a3c",
    "錯過漲幅，且未避開回撤（純損失）":  "#b3261e",
    "躲過跌幅，且避開回撤（純收益）":    "#2e7d32",
    "躲過跌幅，但 MAE 反而更差":       "#6a4c93",
}
LABELS_EN = {
    "錯過漲幅，但避開回撤":            "Missed upside, but less MAE",
    "錯過漲幅，且未避開回撤（純損失）":  "Missed upside, no MAE relief",
    "躲過跌幅，且避開回撤（純收益）":    "Avoided downside + less MAE",
    "躲過跌幅，但 MAE 反而更差":       "Avoided downside, worse MAE",
}

fig, ax = plt.subplots(figsize=(11, 9))
for name, color in COLORS.items():
    sub = cmp[cmp["象限"] == name]
    if len(sub) == 0:
        continue
    ax.scatter(sub["進場價差異%"] * 100, sub["MAE改善(B−C)"] * 100,
               s=90, color=color, alpha=0.85, edgecolor="white",
               label=f"{LABELS_EN[name]} (n={len(sub)})")
for i, r in cmp.iterrows():
    ax.annotate(str(i), (r["進場價差異%"] * 100, r["MAE改善(B−C)"] * 100),
                fontsize=7, xytext=(4, 4), textcoords="offset points")
ax.axhline(0, color="black", lw=1.0)
ax.axvline(0, color="black", lw=1.0)
ax.set_title("RSI layer effect per trade")
ax.set_xlabel("Entry price difference B vs C (%)  →  right = B missed upside")
ax.set_ylabel("MAE improvement B vs C (pp)  →  up = B less painful")
ax.legend(loc="best", fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

C:\Users\king5\AppData\Local\Temp\ipykernel_5136\489800759.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


### 圖 6｜逐年報酬三組對照

In [28]:
x = np.arange(len(yr))
w = 0.27
fig, ax = plt.subplots(figsize=(15, 6))
ax.bar(x - w, yr["A 買進持有"] * 100, w, color="#8a8a8a", label="A: Buy & Hold")
ax.bar(x,     yr["C 只有濾網"] * 100, w, color="#3b6ea5", label="C: Filter only")
ax.bar(x + w, yr["B 完整策略"] * 100, w, color="#b3261e", label="B: Filter + RSI")
ax.axhline(0, color="black", lw=0.8)
ax.set_xticks(x); ax.set_xticklabels(yr.index, rotation=45)
ax.set_title("Annual returns, three groups (2026 is a partial year)")
ax.set_xlabel("Year"); ax.set_ylabel("Return (%)")
ax.legend(); ax.grid(alpha=0.3, axis="y")
plt.tight_layout(); plt.show()

C:\Users\king5\AppData\Local\Temp\ipykernel_5136\972866205.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
## 12. 存檔

四個檔案。`summary_ABC.csv` 用**長格式**（每列一個 組別×期間×指標），
方便階段 7 直接讀取重組。

In [29]:
bt_B = pd.DataFrame({
    "position_B": df["position_B"], "ret_B": df["ret_B"],
    "equity_B": eq_B, "drawdown_B": dd_B,
})
bt_B.index.name = "Date"
bt_B.to_csv(DATA_OUT, date_format="%Y-%m-%d")

trades_B.to_csv(TRADES_OUT, encoding="utf-8-sig")
cmp.to_csv(COMPARE_OUT, encoding="utf-8-sig")

# 長格式彙總表
rows = []
for (w, grp_name) in perf.columns:
    for metric, val in perf[(w, grp_name)].items():
        rows.append({"期間": w, "組別": grp_name, "指標": metric, "值": val})
summary_long = pd.DataFrame(rows)
summary_long.to_csv(SUMMARY_OUT, index=False, encoding="utf-8-sig")

display(summary_long.head(8))

,期間,組別,指標,值
0,全期間,A 買進持有,總報酬率,2.673837
1,全期間,A 買進持有,年化報酬率,0.051148
2,全期間,A 買進持有,年化波動度,0.205211
3,全期間,A 買進持有,Sharpe,0.249247
4,全期間,A 買進持有,MDD（每日）,-0.662204
5,全期間,A 買進持有,MDD（交易層級）,NaN
6,全期間,A 買進持有,MDD 起始日,2000-02-17
7,全期間,A 買進持有,MDD 谷底日,2001-10-03


In [30]:
back_bt  = pd.read_csv(DATA_OUT, index_col="Date", parse_dates=True)
back_tr  = pd.read_csv(TRADES_OUT, index_col="#", encoding="utf-8-sig")
back_cmp = pd.read_csv(COMPARE_OUT, index_col="#", encoding="utf-8-sig")
back_sum = pd.read_csv(SUMMARY_OUT, encoding="utf-8-sig")

ok = (
    len(back_bt) == len(bt_B)
    and np.allclose(back_bt.to_numpy(), bt_B.to_numpy(), equal_nan=True)
    and len(back_tr) == len(trades_B)
    and np.allclose(back_tr["報酬率"].to_numpy(), trades_B["報酬率"].to_numpy())
    and len(back_cmp) == len(cmp)
    and np.allclose(back_cmp["報酬差(B−C)"].to_numpy(), cmp["報酬差(B−C)"].to_numpy())
    and len(back_sum) == len(summary_long)
)

info = pd.DataFrame({"值": [
    f"{DATA_OUT}  ({os.path.getsize(DATA_OUT):,} bytes, {len(back_bt):,} 筆)",
    f"{TRADES_OUT}  ({os.path.getsize(TRADES_OUT):,} bytes, {len(back_tr)} 筆)",
    f"{COMPARE_OUT}  ({os.path.getsize(COMPARE_OUT):,} bytes, {len(back_cmp)} 筆)",
    f"{SUMMARY_OUT}  ({os.path.getsize(SUMMARY_OUT):,} bytes, {len(back_sum)} 列)",
]}, index=["B 組回測", "B 組交易明細", "B vs C 逐筆對照", "三組彙總（長格式）"])
info.index.name = "產出檔案"
display(info)

record("存檔與讀回一致性", ok,
       f"四個檔案皆已寫入，讀回數值完全一致"
       if ok else "讀回的資料與記憶體中不一致，請停止並人工確認")

,值
產出檔案,
B 組回測,"data/backtest_B.csv (422,540 bytes, 6,391 筆)"
B 組交易明細,"data/trades_B.csv (5,106 bytes, 49 筆)"
B vs C 逐筆對照,"data/trade_comparison_BC.csv (13,014 bytes, 4..."
三組彙總（長格式）,"data/summary_ABC.csv (6,235 bytes, 126 列)"


===> [通過] 存檔與讀回一致性
      四個檔案皆已寫入，讀回數值完全一致


---
## 13. 小結

In [31]:
summary = pd.DataFrame(CHECKS)
display(summary)

n_fail = int((summary["結果"] == "異常").sum())
print(f"\n共 {len(summary)} 項驗證，通過 {len(summary) - n_fail} 項，異常 {n_fail} 項")

,驗證項目,結果,說明
0,資料載入一致性,通過,"signals.csv 6,632 筆、backtest_C.csv 6,391 筆、tra..."
1,B 組部位序列,通過,進場 49 次（與 b_entry_day 完全一致）；**每次出場日都與 C 組相同**（...
2,B 組每日報酬序列,通過,"空手日 3,092 天報酬全為 0；單日報酬最大絕對值 6.74%（2009-04-30），..."
3,B 組交易明細,通過,49 筆交易，勝率 42.9%，平均報酬 +3.17%，最差單筆 -11.31%，最深單筆 ...
4,B vs C 逐筆對照,通過,25 筆一對一對照（出場日完全相同）：B 報酬優於 C 者 18 筆（36.7%），報酬差中...
5,A／C 組重現階段 4 結果,通過,逐日比對 equity_A、equity_C、drawdown_A、drawdown_C 四...
6,三組績效比較表,通過,全期間 年化報酬 A 5.11% / C 7.88% / B 4.49%；IS 年化報酬 A...
7,貢獻分解,通過,全期間：濾網貢獻（年化）+2.76%、RSI 貢獻（年化）-3.39%；IS：濾網貢獻（年化...
8,RSI 效果拆解,通過,(a) 等待期間指數變動：錯過漲幅 31 筆、躲過跌幅 18 筆，中位數 +1.38%；(b...
9,逐年報酬三組對照,通過,完整年度 26 年：A 下跌的 7 年三組平均 A -24.79% / C -5.47% /...



共 11 項驗證，通過 11 項，異常 0 項


In [32]:
key = pd.DataFrame(
    {g: [f"{perf[(w, g)][m]:.2%}" if m not in ("Sharpe", "Calmar")
         else f"{perf[(w, g)][m]:.2f}"
         for w in ["全期間", "IS", "OOS"]
         for m in ["年化報酬率", "Sharpe", "MDD（每日）"]]
     for g in ["A 買進持有", "C 只有濾網", "B 完整策略"]},
    index=pd.MultiIndex.from_product([["全期間", "IS", "OOS"],
                                       ["年化報酬率", "Sharpe", "MDD（每日）"]]))
display(key)

extra = pd.DataFrame({"值": [
    f"{contribs['全期間'].loc['年化報酬率', '濾網貢獻 (C−A)']:+.2%} / "
    f"{contribs['全期間'].loc['年化報酬率', 'RSI貢獻 (B−C)']:+.2%}",
    f"{contribs['全期間'].loc['Sharpe', '濾網貢獻 (C−A)']:+.2f} / "
    f"{contribs['全期間'].loc['Sharpe', 'RSI貢獻 (B−C)']:+.2f}",
    f"{contribs['全期間'].loc['MDD（每日）', '濾網貢獻 (C−A)']:+.2%} / "
    f"{contribs['全期間'].loc['MDD（每日）', 'RSI貢獻 (B−C)']:+.2%}",
    f"{int(df['position_C'].sum()):,} → {int(df['position_B'].sum()):,}"
    f"（少 {gap_days:,} 天）",
    f"{int((cmp['報酬差(B−C)'] > 0).sum())} / 25（{(cmp['報酬差(B−C)'] > 0).mean():.1%}）",
    f"{cmp['報酬差(B−C)'].median():+.2%} / {cmp['報酬差(B−C)'].mean():+.2%}",
    f"{int((cmp['MAE改善(B−C)'] > 0).sum())} / 25",
    f"{cmp['C的MAE'].mean():.2%} → {cmp['B的MAE'].mean():.2%}",
    f"{ratio.iloc[-1]:.4f}（{ratio.iloc[-1] - 1:+.2%}）",
]}, index=["全期間 年化報酬：濾網貢獻 / RSI貢獻",
           "全期間 Sharpe：濾網貢獻 / RSI貢獻",
           "全期間 MDD：濾網貢獻 / RSI貢獻",
           "在場天數 C → B",
           "B 報酬優於 C 的筆數",
           "報酬差 中位數 / 平均",
           "B 的 MAE 優於 C 的筆數",
           "平均 MAE：C → B",
           "期末 equity_B / equity_C"])
extra.index.name = "關鍵數字"
display(extra)

display(quad)

A 買進持有   C 只有濾網   B 完整策略
全期間 年化報酬率      5.11%    7.88%    4.49%
    Sharpe      0.25     0.59     0.37
    MDD（每日）  -66.22%  -30.38%  -34.24%
IS  年化報酬率      1.16%    6.75%    3.10%
    Sharpe      0.05     0.49     0.25
    MDD（每日）  -66.22%  -25.73%  -34.24%
OOS 年化報酬率     14.67%   10.46%    7.70%
    Sharpe      0.81     0.82     0.64
    MDD（每日）  -31.63%  -30.38%  -23.71%

,值
關鍵數字,
全期間 年化報酬：濾網貢獻 / RSI貢獻,+2.76% / -3.39%
全期間 Sharpe：濾網貢獻 / RSI貢獻,+0.34 / -0.22
全期間 MDD：濾網貢獻 / RSI貢獻,+35.84% / -3.86%
在場天數 C → B,"3,908 → 3,251（少 657 天）"
B 報酬優於 C 的筆數,18 / 25（36.7%）
報酬差 中位數 / 平均,-1.34% / -1.92%
B 的 MAE 優於 C 的筆數,22 / 25
平均 MAE：C → B,-3.65% → -4.22%
期末 equity_B / equity_C,0.4348（-56.52%）


,筆數,比例,平均報酬差(B−C)
象限,,,
錯過漲幅，且未避開回撤（純損失）,27,55.1%,-4.31%
躲過跌幅，且避開回撤（純收益）,18,36.7%,+1.81%
錯過漲幅，但避開回撤,4,8.2%,-2.61%


### 本階段結論

以下只陳述數字與觀察到的事實，
不評價策略好壞、不建議改進方向、不對原論文下結論（那是階段 7 的工作）。

**規則實作已驗證**

- `regime` 與 `b_entry_day` 皆未再 shift，兩者直接使用：
  B 組進場日與階段 5 的 `b_entry_day` 完全一致（25 次）。
- **B 組出場日與 C 組出場日完全相同**，這是「B 與 C 只差進場時點」的設計前提，
  已嚴格驗證通過，因此 `B − C` 是乾淨的 RSI 效果。
- 每日報酬按進場／持有／出場／空手四類分別套公式，四類互斥且涵蓋全部交易日，
  空手日報酬全為 0。
- `perf_stats()` 與階段 4 完全相同（直接複製，未修改）。重算的 A、C 兩組
  與階段 4 存檔的 `equity_A`、`equity_C`、`drawdown_A`、`drawdown_C`
  **逐日比對**，最大差異小於 1e-10。

**三組績效**

全期間、IS、OOS 的年化報酬、Sharpe、每日 MDD 見上方彙整表，完整指標見區塊 7。
IS 與 OOS 的權益曲線各自從 1.0 起算，兩段不能相加，也不能與全期間直接比較。

**兩層的貢獻分解**

區塊 8 把「濾網貢獻（C−A）」與「RSI 貢獻（B−C）」分開列出，
涵蓋年化報酬、Sharpe、MDD、Calmar、年化波動度、在場時間比例六項，三個期間各一張表。

**B vs C 逐筆對照**

25 筆交易一對一（出場日相同），B 報酬優於 C 的筆數與比例、
報酬差的中位數／平均／最好／最差、以及 MAE 改善的筆數與幅度，見區塊 5。

**RSI 效果的四象限拆解**

區塊 9 把每筆交易依「等待期間指數變動」與「MAE 改善」兩個維度分成四類，
列出各象限的筆數、比例與平均報酬差，散布圖見區塊 11 圖 5。

**累積效果**

區塊 11 圖 3 的 `equity_B / equity_C` 比值序列顯示 RSI 這一層的效果如何隨時間累積，
期末比值與期間最高／最低點已印出。

---

### 產出

- `data/backtest_B.csv` —— position_B、ret_B、equity_B、drawdown_B
- `data/trades_B.csv` —— B 組交易明細
- `data/trade_comparison_BC.csv` —— 25 筆 B vs C 逐筆對照
- `data/summary_ABC.csv` —— 三組 × 三期間完整績效（長格式）

### 下一階段

結論撰寫與對原論文的檢討屬於階段 7。
請先人工確認上述三組績效與貢獻分解，再進行下一階段。